# Track4World - 官方展示效果复现

**目标**: 完全按照官方README的推荐方式，实现前景-背景分离的世界坐标系可视化

**关键配置**:
- 坐标系: `world_depthanythingv3` (世界坐标系)
- 模型: DA3 (更好的深度估计)
- 分割: DINO + SAM2 (前景-背景分离)
- 可视化: `vis_3d_efep_world.py`

**使用流程**:
1. Cell 1: 检查GPU
2. Cell 2: 安装依赖
3. Cell 3: 下载权重 (Track4World + SAM2)
4. Cell 4: 上传视频
5. Cell 5: DINO+SAM2分割 (关键步骤！)
6. Cell 6: Track4World 3d_efep推理
7. Cell 7: 打包下载结果

In [ ]:
# Cell 1: 检查 GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
!nvidia-smi

In [ ]:
# Cell 2: 克隆仓库 + 安装依赖
import os, sys

_flag = "/content/.t4w_official_installed"

if not os.path.exists(_flag):
    print("=== 首次安装：克隆仓库 ===")
    
    # 克隆Track4World (使用最新版本)
    if not os.path.exists("/content/Track4World"):
        !git clone --recurse-submodules https://github.com/TencentARC/Track4World.git /content/Track4World
    
    os.chdir("/content/Track4World")
    
    # 安装PyTorch
    !pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
    
    # 安装Track4World依赖
    !pip install -q -r requirements.txt
    
    # 安装Grounded-SAM-2
    print("\n=== 安装 Grounded-SAM-2 ===")
    os.chdir("/content/Track4World/submodules")
    !pip install -q -e .
    !pip install -q --no-build-isolation -e grounding_dino
    
    # 安装其他依赖
    !pip install -q open3d viser plotly supervision
    
    os.chdir("/content/Track4World")
    open(_flag, "w").close()
    
    print("\n✓ 安装完成！正在重启运行时...")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

else:
    print("=== 重启后：配置环境 ===")
    os.chdir("/content/Track4World")
    sys.path.insert(0, '/content/Track4World')
    sys.path.insert(0, '/content/Track4World/submodules')
    
    print("\n=== 依赖检查 ===")
    for pkg, mod in [
        ('torch','torch'), ('numpy','numpy'), ('cv2','cv2'),
        ('open3d','open3d'), ('viser','viser'), ('supervision','supervision'),
    ]:
        try:
            m = __import__(mod)
            print(f"  ✓ {pkg}: {getattr(m,'__version__','?')}")
        except ImportError:
            print(f"  ✗ {pkg}: 未安装")
    
    print("\n✓ 环境就绪，可继续运行 Cell 3")

In [ ]:
# Cell 3: 下载权重
import os
os.chdir("/content/Track4World")
os.makedirs("checkpoints", exist_ok=True)

# 下载Track4World权重
weights = [
    ("track4world_da3.pth", "https://huggingface.co/TencentARC/Track4World/resolve/main/track4world_da3.pth"),
    ("track4world_moge.pth", "https://huggingface.co/TencentARC/Track4World/resolve/main/track4world_moge.pth"),
]

for name, url in weights:
    path = f"checkpoints/{name}"
    if not os.path.exists(path):
        print(f"下载 {name}...")
        !wget -q --show-progress -O {path} {url}
    else:
        print(f"✓ {name} 已存在")

# 下载SAM2权重
sam2_path = "checkpoints/sam2.1_hiera_large.pt"
if not os.path.exists(sam2_path):
    print("\n下载 SAM2 权重...")
    !wget -q --show-progress -O {sam2_path} https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
else:
    print("✓ SAM2 权重已存在")

print("\n=== 权重文件 ===")
!ls -lh checkpoints/*.pt checkpoints/*.pth 2>/dev/null | awk '{print $9, $5}'

In [ ]:
# Cell 4: 上传视频
from google.colab import files
import os

os.chdir("/content/Track4World")
os.makedirs("input_videos", exist_ok=True)

print("请选择要上传的视频文件 (.mp4)")
uploaded = files.upload()

for fname in uploaded:
    dst = f"input_videos/{fname}"
    with open(dst, "wb") as f:
        f.write(uploaded[fname])
    print(f"✓ 已保存: {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")

VIDEO_PATH = f"input_videos/{list(uploaded.keys())[0]}"
VIDEO_NAME = list(uploaded.keys())[0].replace('.mp4', '')
OUTPUT_DIR = f"results/{VIDEO_NAME}"

print(f"\n视频路径: {VIDEO_PATH}")
print(f"输出目录: {OUTPUT_DIR}")

In [ ]:
# Cell 5: DINO + SAM2 分割 (关键步骤！)
# 这一步会生成动态物体的mask，用于前景-背景分离

import os
os.chdir("/content/Track4World")

# ============ 可修改参数 ============
TEXT_PROMPT = "person. golf ball. golf club."  # 修改为你视频中的动态物体，例如: "cat.", "person.", "car.", "ball."
# ====================================

print(f"正在分割动态物体: {TEXT_PROMPT}")
print(f"输出目录: {OUTPUT_DIR}\n")

cmd = f"""
python scripts/run_dino_sam2.py \
    --video-path {VIDEO_PATH} \
    --sam2-checkpoint checkpoints/sam2.1_hiera_large.pt \
    --output-dir {OUTPUT_DIR} \
    --text-prompt "{TEXT_PROMPT}"
"""

print(f"执行命令:\n{cmd}\n")
!{cmd}

# 验证mask生成
mask_dir = f"{OUTPUT_DIR}/mask"
if os.path.exists(mask_dir):
    mask_count = len([f for f in os.listdir(mask_dir) if f.endswith('.png')])
    print(f"\n✓ 分割完成！生成了 {mask_count} 个mask文件")
else:
    print("\n⚠️ 警告：未找到mask目录，可能分割失败")

In [ ]:
# Cell 6: Track4World 3d_efep 推理 (世界坐标系)

import os
os.chdir("/content/Track4World")

# ============ 可修改参数 ============
IMAGE_SIZE = 448
MAX_FRAMES = 20  # 测试用20帧，完整推理改为-1
# ====================================

print("=== Track4World 3d_efep 推理 ===")
print(f"坐标系: world_depthanythingv3")
print(f"模型: DA3")
print(f"分辨率: {IMAGE_SIZE}")
print(f"帧数: {MAX_FRAMES if MAX_FRAMES > 0 else '全部'}\n")

cmd = f"""
python demo.py \
    --mp4_path {VIDEO_PATH} \
    --coordinate world_depthanythingv3 \
    --mode 3d_efep \
    --Ts {MAX_FRAMES} \
    --ckpt_init checkpoints/track4world_da3.pth \
    --image_size {IMAGE_SIZE} \
    --save_base_dir {OUTPUT_DIR}
"""

print(f"执行命令:\n{cmd}\n")
!{cmd}

# 验证输出
output_3d = f"{OUTPUT_DIR}/3d_efep_output"
if os.path.exists(output_3d):
    ply_count = len([f for f in os.listdir(output_3d) if f.endswith('.ply')])
    print(f"\n✓ 推理完成！生成了 {ply_count} 个点云文件")
    
    # 检查关键文件
    key_files = ['trajectory_all_pointmap.npy', 'c2w.npy']
    for f in key_files:
        path = f"{output_3d}/{f}"
        if os.path.exists(path):
            print(f"  ✓ {f}")
        else:
            print(f"  ✗ {f} (缺失)")
else:
    print("\n⚠️ 警告：未找到输出目录")

In [ ]:
# Cell 7: 打包并下载结果

import os
from google.colab import files

os.chdir("/content/Track4World")

zip_name = f"/content/{VIDEO_NAME}_official_results.zip"

print(f"正在打包: {OUTPUT_DIR}")
print("排除: input_copy.mp4 (节省空间)\n")

!zip -r {zip_name} {OUTPUT_DIR} -x "*/input_copy.mp4"

if os.path.exists(zip_name):
    size_mb = os.path.getsize(zip_name) / 1e6
    print(f"\n✓ 压缩完成: {size_mb:.1f} MB")
    print("正在下载...")
    files.download(zip_name)
    print("\n下载已启动，请查看浏览器下载列表")
else:
    print("\n✗ 打包失败")

## 本地可视化说明

下载结果后，在本地使用以下命令可视化：

```bash
# 解压结果
unzip *_official_results.zip

# 使用世界坐标系可视化 (前景-背景分离)
cd E:/bishe2/Track4World
E:/Conda/envs/track4world/python.exe visualization/vis_3d_efep_world.py \
    --ply_dir ../results/your_video_name/3d_efep_output \
    --save_dir ../recordings/your_video_name

# 浏览器打开: http://localhost:8080
```

**关键特性**:
- 静态背景保持不动
- 动态物体有彩色轨迹
- 可调节轨迹长度、点云大小等参数